In [ ]:
initialize() {
	print("-------Sim Start-------");
	print("Initializing simulation...");
	if (!exists("asexual")){
		//parameter to toggle asexual reproduction
		defineConstant("asexual", F);
	}
	defineConstant("popSize", 250);
	defineConstant("epi", T); // whether to include epistasis between mito and nuc muts
	
	//Where to preload the mutations - mitochondria or the nuclear genome
	defineConstant("preload_location", "mito");
	//WHich mutational profile to use 1: del and ben, 2: only del, 3: no mutations with fitness effects
	defineConstant("mut_profile", 2);
	
	defineConstant("chrom_length", 1e5);
	
	//Enable pedigree tracking
	initializeSLiMOptions(keepPedigrees=T);
	
	if (!asexual){ 
		initializeSex();
	}

If not passed "asexual" from the command line argument, then we set the constant "asexual" as false.

Define the variable "popSize" to 250. 
Define the constant "epi" as True or False. This will determine whether or not we use epistasis in the simulation.

Load in the command line variable for the preload location. This decides which chromosome (mitochondria, or nuclear) we will seed with deleterious mutations.
Load in the command line argument for sets of mutations. These mutations will constantly run during the simulation. This variable is called "mut_profile". If the profile is 1, we use both beneficial and deleterious mutations. If 2, we only use deleterious mutations. If 3, we only use neutral mutations.

Define the chromosome length constant.

Enable pedigree tracking.

If the constant "asexual" is false, we initiate sex.

In [ ]:
//Preloaded mutations types for seeding initial population.
	//Mito
	initializeMutationType("m12", 0.7, "f", -0.09); // preload
	
	//Nuclear
	//changing back to fixed for consistency
	initializeMutationType("m11", 0.7, "f", -0.09); // preload
	//initializeMutationType("m6", 0.3, "f", -0.05); // preload
	
	initializeMutationType("m9", 1.0, "f", 0.0); //synonymous
	initializeMutationType("m10", 1.0, "f", 0.0); //synonymous

We initialize the preloading mutations for the mitochondria and nuclear genomes.
m12 (mt genome) and m11 (nuclear genome) have the same arguments. Both have a dominance coefficience on 0.7, and use a fixed distribution with a mean of -0.09. This will keep the seeded mutations consistent when preloading.

We initialize the neutral mutations, m9 and m10. Both of these have no effect on the individual's fitness, which makes it appropriate for running the simuation without mutations. Making the distribution fixed with a mean of 0.0 achieves this effect.

In [ ]:
//create normal mutation distributions
	if (mut_profile == 1){
		//Both del and ben muts
		
		// Nuclear mutations
		initializeMutationType("m1", 0.5, "g", -0.01, 0.2); // Nuclear negative A
		initializeMutationType("m2", 0.5, "g", -0.01, 0.2); // Nuclear negative B
		initializeMutationType("m3", 0.5, "g", 0.001, 1); // Nuclear positive A
		initializeMutationType("m4", 0.5, "g", 0.001, 1); // Nuclear positive B
		
		// Mitochondrial mutations
		initializeMutationType("m5", 1.0, "g", -0.05, 0.2); // Mitochondrial negative A
		initializeMutationType("m6", 1.0, "g", -0.05, 0.2); // Mitochondrial negative B
		initializeMutationType("m7", 1.0, "g", 0.001, 1); // Mitochondrial positive A
		initializeMutationType("m8", 1.0, "g", 0.001, 1); // Mitochondrial positive B
		
		initializeGenomicElementType("g1", c(m1, m2, m3, m4), c(0.49, 0.49, 0.01, 0.01)); // Nuclear genome
		initializeGenomicElementType("g2", c(m5, m6, m7, m8), c(0.49, 0.49, 0.01, 0.01)); // Mitochondrial genome
	}

If the mutation profile is equal to 1, we will use both deleterious and beneficial mutations.

We have 4 mutations for the nuclear genome and 4 for the mitochondrial genome. Having specific mutations for each genome allows epistasis interactions between the genomes. All of these mutations have a distribution of gamma, to help vary their effects on the population. They also have small means to keep a minimal effect on the individuals' fitness.

Nuclear mutations: m1 and m2 are identical deleterious mutations for the nuclear genome. They have a domincance coefficient of 0.5, a mean of -0.01, and an alpha shape parameter of 0.2.
m3 an m4 are identical beneficial mutations. They have a dominance coefficient of 0.5, a mean of 0.001, and an alpha shapre parameter of 1.

The mitochondrial mutations, m5-m8, are nearly identical to the nuclear. m5 and m6 have a mean of -0.5, which produces a slightly greater effect on fitness.

The genomic element types create groups of mutations and proportions of those mutations. We created g1 for the nuclear mutations, and g2 for the mitochondria. These mutations can only be acquired and mutate within the bounds of the genomic element.

In [ ]:
else if (mut_profile == 2){
		//only del muts
		
		initializeMutationType("m1", 0.5, "g", -0.01, 0.2); // Nuclear negative A
		initializeMutationType("m2", 0.5, "g", -0.01, 0.2); // Nuclear negative B
		initializeMutationType("m3", 0.5, "g", -0.01, 0.2); // Nuclear negative A
		initializeMutationType("m4", 0.5, "g", -0.01, 0.2); // Nuclear negative B
		
		// Mitochondrial mutations
		initializeMutationType("m5", 1.0, "g", -0.05, 0.2); // Mitochondrial negative A
		initializeMutationType("m6", 1.0, "g", -0.05, 0.2); // Mitochondrial negative B
		initializeMutationType("m7", 1.0, "g", -0.05, 0.2); // Mitochondrial negative A
		initializeMutationType("m8", 1.0, "g", -0.05, 0.2); // Mitochondrial negative B
		
		initializeGenomicElementType("g1", c(m1, m2, m3, m4), c(0.49, 0.49, 0.01, 0.01)); // Nuclear genome
		initializeGenomicElementType("g2", c(m5, m6, m7, m8), c(0.49, 0.49, 0.01, 0.01)); // Mitochondrial genome

If the mut_profile is 2, we will only use deleterious mutations.

All four of the nuclear mutations, m1-m4, have a dominance coefficient of 0.5, using a gamma distribution, with a mean of -0.01 and an alpha shape parameter of 0.2.

The mitochondrial mutations, m5-m8, are slightly stronger, increasing their dominance coefficient to 1.0 and their mean to -0.5

Like the genomic elements before, these place the nuclear and mitochindria mutations in their respective groups and marking the ratios.

In [ ]:
} else if (mut_profile == 3){
		//no muts (techincially synonymous muts with no fitness effect)
		//Nuclear
		initializeMutationType("m1", 0.5, "f", 0.0); // Nuclear negative A
		initializeMutationType("m2", 0.5, "f", 0.0); // Nuclear negative B
		initializeMutationType("m3", 0.5, "f", 0.0); // Nuclear negative A
		initializeMutationType("m4", 0.5, "f", 0.0); // Nuclear negative B
		
		// Mitochondrial mutations
		initializeMutationType("m5", 1.0, "f", 0.0); // Mitochondrial negative A
		initializeMutationType("m6", 1.0, "f", 0.0); // Mitochondrial negative B
		initializeMutationType("m7", 1.0, "f", 0.0); // Mitochondrial negative A
		initializeMutationType("m8", 1.0, "f", 0.0); // Mitochondrial negative B
		
		initializeGenomicElementType("g1", c(m9), c(1));
		//autosomal
		initializeGenomicElementType("g2", c(m10), c(1));
	}

Profile 3 has no positive or beneficial mutations.

m1-m8 are placeholders to avoid errors down the line. The mean for each is 0.0, giving them no effect.

Regardless, the neutral mutations, m9 and m10 that we initialized earlier, are the only mutations entered into g1 and g2. This means they are the only mutations in the simulation, aside from m11 and m12, the preloading mutations.

In [ ]:
    m1.convertToSubstitution = F;
	m2.convertToSubstitution = F;
	m3.convertToSubstitution = F;
	m4.convertToSubstitution = F;
	m5.convertToSubstitution = F;
	m6.convertToSubstitution = F;
	m7.convertToSubstitution = F;
	m8.convertToSubstitution = F;
	m9.convertToSubstitution = F;
	m10.convertToSubstitution = F;
	m11.convertToSubstitution = F;
	m12.convertToSubstitution = F;

This segment keeps all of the mutations from fixing, or being removed from the fitness calculation once the whole populations has it. This will keep our fitness rates consistent throughout the simulation.

In [ ]:
    defineConstant("mito_chrom_length", 1e5);
	defineConstant("nuc_chrom_length", 1e5);
	
	//chromosome 1: mitochondrial chromosome
	//adding in chromosome length
	initializeChromosome(1,mito_chrom_length, type="HF");
	initializeGenomicElement(g1, 0, mito_chrom_length-1);
	initializeMutationRate(1e-6);
	initializeRecombinationRate(0.0);
	
	//chromosome 2: autosome
	//adding chromosome length
	initializeChromosome(2,nuc_chrom_length,type="A");
	initializeGenomicElement(g2, 0, nuc_chrom_length-1);
	initializeMutationRate(1e-7);
	initializeRecombinationRate(1e-7);

}

We establish two constants for both chromosomes' length.

We establish the mitochondrial chromosome, as chromosome 1, with the variable's length established just above, and the type HF. HF is a halpoid that is maternally inherited, much like the mitochondrial genome.
The genomic element g1 is added, and takes up the whole of the chromosome.
Chromosome 1 has a mutation rate of 1e-6, which is higher than nuclear's.
Chomosome 1's recombination rate is set to 0.0.

Chromosome 2 is created in the same way. It is established as an autosome instead of an HF, and uses the nuc_chrom_length established above.
G2 is initialized.
The mutation rate is lower than chromosome 1.
The recombination rate is set to 1e-7 to allow some recombination.

In [ ]:
1 early() {
	
	//initialize population
	sim.addSubpop("p1", popSize);
	//Asexual populations reproducte by cloning 100% of the time
	if (asexual == T){
		p1.setCloningRate(1.0);
	} else {
		p1.setCloningRate(0.0);
	}
	
	//Vector to store fitness over time
	p1.setValue("fitness_over_time", c());
}

"1 early()" means this is run at the beginning of the first cycle.

We initialize the population and call it "p1".

If the asexual variable is true, then we set the cloning rate to 1.0.
Otherwise, it is set to 0.0, since no one in the population is reproducting asexually.

A vector to store p1's fitness over time is estaished.

In [ ]:
//seeds deleterious mutations
1 modifyChild() {
	//gather mitochondrial chromosomes
	
	mut_chroms = c();
	for (hap in child.haplosomes){
		if (preload_location == "mito" ){
			if (hap.chromosome.id == 1){
				mut_chroms = c(mut_chroms, hap);
			}
		} else {
			if (hap.chromosome.id == 2){
				mut_chroms = c(mut_chroms, hap);
			}
		}
	}

"1 modifyChild()" modifies the offspring for the first generation.

To begin we create a vector to hold the chromosomes we wish to modify.
For each haplsome in each child's genome (this includes the two haplosomes that make up chromosome 2), if the variable preload_location is "mito" we gather the haplosomes of chromosome 1 into the list.
If it is "nucl", we gather the same from chromosome 2.

In [ ]:
if (preload_location == "mito"){
		for (i in 1:20){
			//seed deleterious mutations into mitochondrial chromosomes
			mut_chroms.addNewDrawnMutation(m12, rdunif(1,0,chrom_length));
			//mut_chroms.addNewDrawnMutation(m6, rdunif(1,0,chrom_length));
		}
	} else {
		for (i in 1:20){
			//seed deleterious mutations into nuclear chromosomes
			//mut_chroms.addNewDrawnMutation(m11, rdunif(1,0,chrom_length));
			mut_chroms.addNewDrawnMutation(m2, rdunif(1,0,chrom_length));
		}
	}
	return T;
}

If preload_location is "mito" each haplosome in the mut_chroms vector will be given m12 20 times via the for loop.
Otherwise, we will seed m11 into each of the nuclear haplsomomes 20 times.

Finally, we return True to allow all of the low fitness offspring into the simulation.

In [ ]:
1: late() {
	
	inds = sim.subpopulations.individuals;
	popFitt = c();
	
	// loop through each individual in the population to calculate fitness
	for (ind in inds) {
		globalFitness = 0;
		unique = ind.uniqueMutations; // get the unique mutations for the individual
		uniqueNuc = unique[unique.mutationType == m1 | unique.mutationType == m2 | unique.mutationType == m3 | unique.mutationType == m4 | unique.mutationType == m11]; // subset to get only nuclear mutations
		uniqueMito = unique[unique.mutationType == m5 | unique.mutationType == m6 | unique.mutationType == m7 | unique.mutationType == m8| unique.mutationType == m12]; // subset to get only mitochondrial mutations

At the end of each generation, we calculate the fitness for the populations.

First, we gather all of the individuals and locate the unique mutations for each individual.

In [ ]:
// loop through each mitochondrial mutation to calculate the epistatic interactions with nuclear mutations
		for (m in uniqueMito) {
			if (!epi) {
				globalFitness = globalFitness + (m.selectionCoeff * abs(m.selectionCoeff));
			} else {
				total = 0;
				epistatic = uniqueNuc[uniqueNuc.position % 125 == m.position % 125]; // get the nuclear mutations that are epistatic with the mitochondrial mutation (i.e., those that occur at the same position modulo 125)

We loop through each unique mutation in the mitochondrial genome.
If epi has not been activated, we add to the global fitness the selection coefficient of the mutation squared while preserving its sign. 
If epi has been activated, then we gather all potentially epistatic nuclear mutations as determined by the epiSt argument. Mutations are epistatic if the position of the nuclear mutation % epiSt is equal to the position of the mitochondrial mutation % epiSt.

In [ ]:
// loop through the epistatic nuclear mutations to calculate the total effect on fitness
				for (n in epistatic) {
					if (n.mutationType == m1 | n.mutationType == m3) {
						total = total + abs(n.selectionCoeff);
					
					}
					else {total = total - abs(n.selectionCoeff);}
				}

Likewise, ee loop through the epistatic nuclear mutations to calculate the total effect on fitness. 

If the nuclear mutations are m1 or m3 (type A mutations), then the absoluate value of the selection coefficient is added to the total epistatic effect for this mitochondrial mutation site. 

If nuclear mutations are m2 or m4, the absolute value of the selection coefficient is subtracted from the total.

In [ ]:
// If there are no epistatic interactions, add the effect of the mitochondrial mutation to fitness
				
				//			globalFitness = globalFitness + (m.selectionCoeff * abs(m.selectionCoeff));
				
				if (length(epistatic) == 0) {
					globalFitness = globalFitness + (m.selectionCoeff * abs(m.selectionCoeff));
				}
				
				if (m.mutationType == m6 | m.mutationType == m8) {
					globalFitness = globalFitness - (total * abs(m.selectionCoeff));
				}
				
				else
				{globalFitness = globalFitness + (total * abs(m.selectionCoeff));
				}
			}
		}

If there are no epistatic nuclear mutations, the fitness is calculated normally as the selection coefficient squared while keeping the same sign. 

If the mitochondrial mutation is m6 or m8 (type B), then the total epistatic effect times the absolute values of the mitochondrial mutation's selection coefficient is subtracted from the current global fitness. 

Otherwise, if the mitochondrial mutation is type A, then the total epistatic effect times the absolute value of the selection coefficient is added to the current global fitness.

In [ ]:
// loop through the nuclear mutations to add their effects on fitness (after accounting for epistasis with mitochondrial mutations)
		for (mut in uniqueNuc) {
			globalFitness = globalFitness + (abs(mut.selectionCoeff) * mut.selectionCoeff);
		}

Loop through the nuclear mutations and account for epistasis with the mitochondiral mutations, before adding their effects to fitness. 

In [ ]:
// calculate the final fitness for the individual, ensuring it is not negative
		Fitness = max(0.0, 1.0 + globalFitness * 3.0); // scale fitness to ensure it is positive and has a reasonable range
		popFitt = c(popFitt, Fitness);
	}
	
	inds.fitnessScaling = popFitt;
}

For each individual, we scale their individual fitness according to a linear fitness to ensure that the majority of fitness values are between 0 and 2. 

We then add the individual fitness to an array of all individuals' fitness. FitnessScaling is applied to each individual.

In [ ]:
mutationEffect(m1) {
	return 1.0;
}

mutationEffect(m2) {
	return 1.0;
}

mutationEffect(m3) {
	return 1.0;
}

mutationEffect(m4) {
	return 1.0;
}

mutationEffect(m5) {
	return 1.0;
}

mutationEffect(m6) {
	return 1.0;
}

mutationEffect(m7) {
	return 1.0;
}

mutationEffect(m8) {
	return 1.0;
}

mutationEffect(m9) {
	return 1.0;
}

mutationEffect(m10) {
	return 1.0;
}

mutationEffect(m11) {
	return 1.0;
}

mutationEffect(m12) {
	return 1.0;
}

For each mutation, we set its mutationEffect to return 1.0. This deactivates SLiM's default mutation effect and uses our previous calculations instead.

This allows us to correctly account for epistasis.

In [ ]:
2 early() {
	//Mutational metdown fitness
	print("Mean fitness at generation 1:");
	print(mean(p1.cachedFitness(NULL)));

At the beginning of generation 2, we print out the mean fitness of the population.

In [ ]:
2: early() {
	if (mean(p1.cachedFitness(NULL)) >= 1.0){
		//Send a message when threshold is reached
		print("Mean fitness at generation " + sim.cycle + " has reached 1.0");
		print("Mean fitness at generation " + sim.cycle + ":");
		print(mean(p1.cachedFitness(NULL)));
		
		if (asexual == T){
			writeFile(getwd() + "/asexual_cycles_to_escape.txt", asString(sim.cycle), append=T);
		} else {
			writeFile(getwd() + "/sexual_cycles_to_escape.txt", asString(sim.cycle), append=T);
		}
		
		//deregister this script block to stop this check print
		community.deregisterScriptBlock(self);
	}
}

For each generation, two and onwards, we check if the average fitness level has reached 1.0 (neutral). This would mean the population has recovered from the seeded mutations.

We then write the cycle it escaped to respective files for sexual and asexual.

Finally, we deregister the script so it no longer runs this check.

In [ ]:
2000 late() {
	//sim.simulationFinished(); 
	//sim.outputFull();
}

We should probably just delete this code.

In [ ]:
2001 early() {
	//Final fitness - output
	print("Mean fitness at generation 2000:");
	print(mean(p1.cachedFitness(NULL)));
	if (asexual == T){
		writeFile(getwd() + "/asexual_final_fitness.txt", asString(mean(p1.cachedFitness(NULL))), append=T);
	} else {
		writeFile(getwd() + "/sexual_final_fitness.txt", asString(mean(p1.cachedFitness(NULL))), append=T);
	}
}

At the beginning of the 2001 generation, we print the final fitness and write it to its respecive file.

In [ ]:
2:2001 early() {
	//update stored data about fitness over time for this simulation
	f = mean(p1.cachedFitness(NULL));
	
	v = p1.getValue("fitness_over_time");
	v = c(v, f);
	p1.setValue("fitness_over_time", v);
}

For the beginning of each generation from 2 - 2001, we update the mean fitness, and add it to the "fitness_over_time" vector.

In [ ]:
2001 late() {
	//Export fitness over time data for this simulation
	v = p1.getValue("fitness_over_time");
	s = paste(v, sep=" ");
	if (asexual == T)
		writeFile(getwd() + "/asexual_fitness_over_time.txt", s, append=T);
	else
		writeFile(getwd() + "/sexual_fitness_over_time.txt", s, append=T);
	
	print(length(v) + " entries exported");
	print("Simulation complete");
}

At the end of the 2001 generation, we write the "fitness_over_time" vector to its respective file, either asexual or sexual.

We then print the number of entries exported and that the simulation is complete.